# Missing Data Handling

NIPALS estimates scores iteratively, skipping missing elements. The NLP method solves an optimization problem — more accurate but requires IPOPT or falls back to the NEOS server.

In [1]:
import pandas as pd
import numpy as np
import pyphi.calc as phi
import pyphi.plots as pp
from bokeh.io import output_notebook
output_notebook(hide_banner=True)
import pyphi.plots as _ppmod
_ppmod.output_file = lambda *a, **kw: None


Will be using the NEOS server in the absence of IPOPT and GAMS


## Load Data with Missing Values

In [2]:
features_md = pd.read_excel('../data/Automobiles PCA w MD.xlsx', 'Features',
                            na_values=np.nan, engine='openpyxl')
classid     = pd.read_excel('../data/Automobiles PCA w MD.xlsx', 'CLASSID',
                            na_values=np.nan, engine='openpyxl')

# Count missing values per variable
data_cols = features_md.iloc[:, 1:]
print('Missing values per variable:')
print(data_cols.isna().sum()[data_cols.isna().sum() > 0])


Missing values per variable:
Cylinders       34
Displacement    43
Horsepower      42
Model Year      19
Weight          40
dtype: int64


## NIPALS (default) — handles missing data natively

In [3]:
pcaobj_nipals = phi.pca(features_md, 3, md_algorithm='nipals')
print('NIPALS loadings (PC1):', pcaobj_nipals['P'][:, 0])
pp.score_scatter(pcaobj_nipals, [1, 2], CLASSID=classid, colorby='Cylinders')


phi.pca using NIPALS executed on: 2026-03-26 23:55:03.271048
# Iterations for PC #1:  14
# Iterations for PC #2:  17
# Iterations for PC #3:  48
--------------------------------------------------------------
PC #      Eig        R2X       sum(R2X) 
PC #1:      3.930    0.771     0.771
PC #2:      1.129    0.173     0.944
PC #3:      0.244    0.031     0.975
--------------------------------------------------------------
NIPALS loadings (PC1): [ 0.4808776   0.49333027  0.48097443 -0.24946913  0.48146727]


## Complete-Data Baseline

In [4]:
features_complete = pd.read_excel('../data/Automobiles PLS.xlsx', 'Features',
                                  na_values=np.nan, engine='openpyxl')
pcaobj_complete = phi.pca(features_complete, 3)
print('Complete-data loadings (PC1):', pcaobj_complete['P'][:, 0])


phi.pca using NIPALS executed on: 2026-03-26 23:55:03.425385
# Iterations for PC #1:  10
# Iterations for PC #2:  8
# Iterations for PC #3:  35
--------------------------------------------------------------
PC #      Eig        R2X       sum(R2X) 
PC #1:      3.871    0.777     0.777
PC #2:      0.818    0.164     0.941
PC #3:      0.172    0.032     0.973
--------------------------------------------------------------
Complete-data loadings (PC1): [ 0.4837139   0.49640422  0.47571761 -0.24916345  0.48084722]


## Effect of Missing Variables on Predictions

When `Xnew` contains `np.nan`, NIPALS-based prediction still works but SPE/T² values may diverge from the stored training values. This is a known limitation.

In [5]:
# Introduce a missing value in a new observation
X_new_with_nan = features_complete.iloc[-5:].copy().reset_index(drop=True)
X_new_with_nan.iloc[0, 2] = np.nan  # blank out one variable
pred = phi.pca_pred(X_new_with_nan, pcaobj_complete)
print('Tnew for rows with/without NaN:')
print(pred['Tnew'])


Tnew for rows with/without NaN:
[[-1.2863966  -1.35280369  0.37789578]
 [-1.42170693 -1.31351446  0.30142827]
 [-1.75091714 -1.18973787  0.29703919]
 [-1.69589551 -1.2457083   0.1953402 ]
 [-1.60988293 -1.26897489  0.25463302]]


## NLP Algorithm (requires IPOPT or NEOS server)

Set `md_algorithm='nlp'` to use the NLP-based estimator. Requires either IPOPT in your system PATH or a valid `NEOS_EMAIL` environment variable.

In [6]:
import os
ipopt_available = bool(__import__('shutil').which('ipopt'))
neos_configured = bool(os.environ.get('NEOS_EMAIL'))

if ipopt_available or neos_configured:
    try:
        pcaobj_nlp = phi.pca(features_md, 3, md_algorithm='nlp')
        print('NLP loadings (PC1):', pcaobj_nlp['P'][:, 0])
    except (TypeError, RuntimeError) as e:
        print(f'NLP failed: {type(e).__name__}: {str(e)[:100]}')
        print('(This may be due to environment/dependency issues)')
else:
    print('Skipping NLP: IPOPT not found and NEOS_EMAIL not set.')
    print('To use NLP: install IPOPT or set NEOS_EMAIL=your@email.com')


phi.pca using NLP with Ipopt executed on: 2026-03-26 23:55:03.510508
ERROR: Rule failed when generating expression for Constraint c20b with index
(np.int64(1), np.int64(1)): TypeError: Calling np.sum(generator) is
deprecated.Use np.sum(np.fromiter(generator)) or the python sum builtin
instead.


ERROR: Constructing component 'c20b' from data=None failed:
        TypeError: Calling np.sum(generator) is deprecated.Use
        np.sum(np.fromiter(generator)) or the python sum builtin instead.


NLP failed: TypeError: Calling np.sum(generator) is deprecated.Use np.sum(np.fromiter(generator)) or the python sum builtin
(This may be due to environment/dependency issues)
